# Ejercicio 6 — Contrastes ortogonales L/Q completos en un $3^2$

**Objetivo.** Reproducir con código la tabla de contrastes ortogonales de la teoría
(componentes lineal y cuadrático de cada factor, más los cuatro componentes de la
interacción bilineal $A_LB_L$, $A_LB_Q$, $A_QB_L$, $A_QB_Q$), verificar su ortogonalidad,
calcular la suma de cuadrados de cada uno y contrastarlas con el ANOVA de `statsmodels`.
Finalmente, agrupar (pool) los componentes de orden superior no significativos como
estimador del error, ya que el $3^2$ sin réplicas es un diseño saturado (8 corridas
independientes para 8 efectos).

**Factores:**
- $A$ = Presión de sellado: 2.0 (−1), 2.5 (0), 3.0 bar (+1)
- $B$ = Temperatura de sellado: 120 (−1), 140 (0), 160 °C (+1)

**Respuesta:** Resistencia del sellado (N/15mm)

**Dataset:** `../../datos/sellado-empaques-3k.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats

df = pd.read_csv('../../datos/sellado-empaques-3k.csv')
df = df.sort_values(['x1', 'x2']).reset_index(drop=True)
print(f'Corridas: {len(df)} (3^2 = 9 sin réplica)')
print(df)

## 1. Matriz de contrastes ortogonales del $3^2$

Con las corridas ordenadas como $(x_1,x_2)=(-1,-1),(-1,0),(-1,1),(0,-1),\dots,(1,1)$,
construimos las columnas $A_L$, $A_Q$, $B_L$, $B_Q$ (teoría §4.1) y las cuatro columnas
de interacción como productos elemento a elemento.

In [ ]:
AL = np.array([-1, -1, -1,  0, 0, 0,  1, 1, 1])
AQ = np.array([ 1,  1,  1, -2,-2,-2,  1, 1, 1])
BL = np.array([-1,  0,  1, -1, 0, 1, -1, 0, 1])
BQ = np.array([ 1, -2,  1,  1,-2, 1,  1,-2, 1])

contrastes = {
    'A_L': AL, 'A_Q': AQ, 'B_L': BL, 'B_Q': BQ,
    'A_LB_L': AL*BL, 'A_LB_Q': AL*BQ, 'A_QB_L': AQ*BL, 'A_QB_Q': AQ*BQ,
}

# Verificación de ortogonalidad: el producto interno de cualquier par debe ser 0
nombres = list(contrastes.keys())
ortogonal = True
for i in range(len(nombres)):
    for j in range(i+1, len(nombres)):
        dp = np.dot(contrastes[nombres[i]], contrastes[nombres[j]])
        if dp != 0:
            ortogonal = False
            print(f'⚠ NO ortogonal: {nombres[i]} · {nombres[j]} = {dp}')
print('Todos los pares de contrastes son ortogonales:' , ortogonal)

## 2. Suma de cuadrados de cada contraste

Con $n=1$ observación por celda (sin réplica), la suma de cuadrados de cada efecto es
$SC = C^2 / (n\sum c_i^2)$, donde $C=\sum_i c_i y_i$ (teoría §4).

In [ ]:
y = df['resistencia'].values
n = 1  # una observación por celda

filas = []
for nombre, c in contrastes.items():
    C = np.dot(c, y)
    sum_c2 = np.dot(c, c)
    SC = C**2 / (n * sum_c2)
    filas.append([nombre, round(C, 3), sum_c2, round(SC, 4)])

tabla = pd.DataFrame(filas, columns=['Efecto', 'C', 'Σc_i²', 'SC'])
print(tabla)

SST = np.sum((y - y.mean())**2)
print(f'\nSuma de las 8 SC:     {tabla["SC"].sum():.4f}')
print(f'SST (8 gl, corregida): {SST:.4f}')
print('Los 8 contrastes ortogonales reparten exactamente la variabilidad total.')

## 3. Comparación con el ANOVA de `statsmodels`

Ajustamos el modelo reducido (L, Q y la interacción bilineal $A_LB_L$) y comparamos su
tabla tipo I con las SC calculadas manualmente para esos mismos cinco efectos.

In [ ]:
modelo = smf.ols('resistencia ~ x1 + x2 + I(x1**2) + I(x2**2) + x1:x2', data=df).fit()
anova1 = sm.stats.anova_lm(modelo, typ=1)
print(anova1.round(4))
print()
print('Compare esta columna sum_sq con SC de A_L (x1), B_L (x2), A_Q (I(x1**2)),')
print('B_Q (I(x2**2)) y A_LB_L (x1:x2) en la tabla de la sección 2: son idénticas.')

## 4. Agrupar componentes de orden superior como error

El $3^2$ sin réplicas tiene 8 corridas independientes (tras remover la media) para 8
posibles efectos: no queda ningún grado de libertad para el error si se estiman los ocho.
En la práctica se asume que las interacciones de orden más alto
($A_QB_Q$, $A_LB_Q$, $A_QB_L$) son despreciables y se **agrupan (pool)** como estimador
del error puro (3 gl), lo que permite construir pruebas F para los cinco efectos de
interés: $A_L$, $A_Q$, $B_L$, $B_Q$ y $A_LB_L$.

In [ ]:
tabla_idx = tabla.set_index('Efecto')
componentes_pool = ['A_QB_Q', 'A_LB_Q', 'A_QB_L']
gl_error = len(componentes_pool)
SCE = tabla_idx.loc[componentes_pool, 'SC'].sum()
MSE = SCE / gl_error
print(f'SC agrupada (error): {SCE:.4f}  con {gl_error} gl  →  MSE = {MSE:.4f}')

print(f'\n{"Efecto":<10}{"SC":>10}{"F":>10}{"p-valor":>10}')
for efecto in ['A_L', 'A_Q', 'B_L', 'B_Q', 'A_LB_L']:
    SC = tabla_idx.loc[efecto, 'SC']
    F = SC / MSE
    p = 1 - stats.f.cdf(F, 1, gl_error)
    print(f'{efecto:<10}{SC:>10.4f}{F:>10.3f}{p:>10.4f}')

## 5. Visualización de la interacción $A_LB_L$

Si las líneas de resistencia vs. $x_2$ (para cada nivel de $x_1$) no son paralelas, la
interacción bilineal es real — coherente con el $SC_{A_LB_L}$ significativo hallado arriba.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for x1_val, sub in df.groupby('x1'):
    sub = sub.sort_values('x2')
    ax.plot(sub['x2'], sub['resistencia'], 'o-', label=f'$x_1$={x1_val}')
ax.set_xlabel('$x_2$ (Temperatura de sellado)')
ax.set_ylabel('Resistencia del sellado (N/15mm)')
ax.set_title('Interacción $A_LB_L$: líneas no paralelas ⇒ interacción real')
ax.legend(title='Presión ($x_1$)')
plt.tight_layout()
plt.show()

## 6. Conclusión

- Los ocho contrastes ortogonales del $3^2$ ($A_L$, $A_Q$, $B_L$, $B_Q$ y los cuatro
  componentes de la interacción) reparten exactamente la SST — es la misma descomposición
  que hace `statsmodels` con `I(x1**2)`, `I(x2**2)` y `x1:x2`, solo que aquí se hizo
  explícita la maquinaria de contrastes de la teoría (§4).
- Con un $3^2$ sin réplicas no hay grados de libertad "gratis" para el error: hay que
  **agrupar** las componentes de interacción de orden más alto (aquellas sin
  justificación física de ser grandes) para poder construir pruebas F.
- En este proceso de sellado, tanto $A_L$ y $B_L$ (efectos lineales) como $A_Q$ y $B_Q$
  (curvatura) son significativos, y la interacción $A_LB_L$ también lo es: la superficie
  de respuesta tiene curvatura real en ambos factores y no es aditiva.
- **Siguiente paso:** con curvatura e interacción confirmadas, se justifica un CCD (o el
  $3^2$ aumentado del Ejercicio 3) para localizar el punto óptimo con precisión.